# Tsetlin bake-off on a free Colab GPU

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Stay on the tab — free Colab disconnects after ~90 min idle and wipes state.
The whole run is ~5–7 min.

In [ ]:
# 1. GPU check + setup (one cell so a re-run after a disconnect just works)
!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'
import os
if not os.path.isdir('tsetlin-market-lab'):
    !git clone --depth 1 https://github.com/naibwedi/tsetlin-market-lab.git
os.chdir('/content/tsetlin-market-lab')
!pip -q install 'numpy<2' 'scikit-learn==1.5.2' pandas pyarrow pyyaml python-dotenv xgboost lightgbm tmu pycuda 2>&1 | tail -2
print('cwd', os.getcwd())

In [ ]:
# 2. Build features (synthetic until real odds are collected) + run the 7 baselines
import glob
if not glob.glob('data/features/X.parquet'):
    !python -m src.ingest.make_synthetic --n-matches 80
    !python -m src.panel.build_panel --config config/features.yaml
    !python -m src.features.booleanize --config config/features.yaml
!python -m src.models.bakeoff --config config/bakeoff.ci.yaml
print('\n' + open('results/summary.md').read())

In [ ]:
# 3. Train the Tsetlin Machine on the GPU
import json, time, numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from tmu.models.classification.vanilla_classifier import TMClassifier
from src.common.config import load_yaml
from src.models.bakeoff import load_split, score

sp = load_split(load_yaml('config/bakeoff.yaml'))
Xtr = sp.X[sp.tr | sp.va].astype(np.uint32); ytr = sp.y[sp.tr | sp.va].astype(np.uint32)
Xte = sp.X[sp.te].astype(np.uint32); yte = sp.y[sp.te].astype(int)
print('train', Xtr.shape, ' test', Xte.shape, ' pos rate', round(float(yte.mean()), 3))

T = 32
try:
    tm = TMClassifier(number_of_clauses=1000, T=T, s=5.0, weighted_clauses=True, platform='CUDA', seed=0)
    backend = 'CUDA'
except Exception:
    tm = TMClassifier(number_of_clauses=1000, T=T, s=5.0, weighted_clauses=True, seed=0)
    backend = 'CPU'
print('backend:', backend)

t0 = time.time()
for epoch in range(40):
    tm.fit(Xtr, ytr)
    if (epoch + 1) % 8 == 0:
        _, cs = tm.predict(Xte, return_class_sums=True)
        cs = np.asarray(cs, float); proba = 1 / (1 + np.exp(-(cs[:, 1] - cs[:, 0]) / T))
        print(f'epoch {epoch+1:2d}  AUC={roc_auc_score(yte, proba):.3f}  '
              f'PR-AUC={average_precision_score(yte, proba):.3f}  ({time.time()-t0:.0f}s)')

m = score(sp.y[sp.te], proba, 0.1)
print('\n=== Tsetlin Machine ===')
print(json.dumps({k: round(v, 4) for k, v in m.items()}, indent=2))

In [ ]:
# 4. The clauses it learned
n_lit = len(sp.feat)
for cls in (1, 0):
    print('\n=== predicts', 'MOVE' if cls else 'NO-MOVE', '===')
    shown = 0
    for c in range(tm.number_of_clauses):
        lits = [sp.feat[k] for k in range(n_lit) if tm.get_ta_action(clause=c, ta=k, the_class=cls)]
        lits += ['NOT ' + sp.feat[k] for k in range(n_lit) if tm.get_ta_action(clause=c, ta=k + n_lit, the_class=cls)]
        if lits and len(lits) <= 5 and shown < 15:
            print('IF', ' AND '.join(lits)); shown += 1